# spinelab — анализ МРТ позвоночника в Colab

**Это исследовательский инструмент, а не диагноз.** Всё, что он выдаёт, требует проверки
врачом-рентгенологом на исходных изображениях. Отдельные этапы — эвристики по яркости
пикселей; в отчёте они помечены отдельно и не являются находками.

## Что делает
DICOM → NIfTI → сегментация (SPINEPS, TotalSpineSeg, TotalSegmentator MRI) →
совмещение серий → измерения (геометрия, мышцы, канал, диски, фасеточные зоны) →
один HTML-отчёт с уровнями доказательности.

На A100 (Colab Pro) имеет смысл профиль **QUALITY** в ячейке 3: он не меняет
определения измерений, а тратит GPU-время на то, чтобы знать их надёжность —
вторая независимая модель позвонков даёт согласие по уровням, а зеркальная TTA
проверяет, устойчиво ли модель определяет лево/право. Последнее прямо относится к
вопросу «слева или справа»: если определение стороны неустойчиво, все сравнения
сторон в этом прогоне недействительны, и отчёт это скажет.

## Порядок работы (4 ячейки, слабый интернет учтён)
1. **Подключить Drive и кэш** — веса моделей (~6–8 ГБ) скачиваются один раз и живут в Drive.
   После обрыва соединения повторный запуск ничего не докачивает.
2. **Установить окружение** — одна идемпотентная ячейка. Ядро не перезапускается принудительно.
3. **Запустить пайплайн** — по этапам, с продолжением с места обрыва (`--force` для пересчёта).
4. **Посмотреть отчёт** — HTML открывается здесь же и копируется в Drive.

## Данные пациента
Положите ZIP с DICOM **в Drive**, а не в публичный git-репозиторий: в заголовках DICOM
есть ФИО, дата рождения и учреждение. Ячейка 3 печатает, какие идентифицирующие теги
нашлись; `python -m spinelab deid` делает деидентифицированную копию.

> Runtime → Change runtime type → **GPU** (T4 хватает; A100/H100 быстрее).

In [1]:
# @title 1 · Drive, кэш весов и исходники {display-mode:"form"}
DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
CACHE_DIR  = "/content/drive/MyDrive/spinelab-cache"  # @param {type:"string"}
BRANCH     = "qa/2026-07-refactor"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}

import os, subprocess, sys
from pathlib import Path

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/omarnuri/MRI-reseqrch.git"
REPO_DIR = Path("/content/spinelab-src")

# Sparse, blobless clone: the repository still contains a ~35 MB DICOM archive and
# a 1.7 MB notebook with embedded outputs, and neither is needed to run anything.
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "init", "--no-cone"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set",
                    "/spinelab/*", "/tests/*", "/docs/*", "/pyproject.toml", "/README.md"],
                   check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, str(REPO_DIR))
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

sha = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"spinelab source: {REPO_DIR} @ {sha}")
print(f"weights cache:   {CACHE_DIR}")
for name in ("spineps", "totalsegmentator", "totalspineseg", "huggingface"):
    d = Path(CACHE_DIR) / "weights" / name
    size_gb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e9 if d.exists() else 0.0
    print(f"  cached {name:18s} {size_gb:5.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
spinelab source: /content/spinelab-src @ 8124484
weights cache:   /content/drive/MyDrive/spinelab-cache
  cached spineps             0.00 GB
  cached totalsegmentator    0.00 GB
  cached totalspineseg       0.00 GB
  cached huggingface         0.00 GB


In [2]:
# @title 2 · Окружение (идемпотентно, без перезапуска ядра) {display-mode:"form"}
INSTALL_SEGMENTATION = True  # @param {type:"boolean"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import importlib, subprocess, sys
from pathlib import Path

def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    if p.returncode != 0:
        print("   !", (p.stderr or p.stdout).strip().splitlines()[-1][:200])
    return p.returncode == 0

def pip(*pkgs, upgrade=False):
    args = [sys.executable, "-m", "pip", "install", "-q"]
    if upgrade:
        args.append("--upgrade")
    return sh(*args, *pkgs)

MARKER = Path("/content/.spinelab_env_ok")

print("apt: dcm2niix, unzip")
sh("apt-get", "-qq", "update")
sh("apt-get", "-qq", "install", "-y", "dcm2niix", "unzip")

if MARKER.exists() and not FORCE_REINSTALL:
    print("python packages: already installed in this VM (tick FORCE_REINSTALL to redo)")
else:
    print("python: core I/O")
    # SimpleITK is what the `register` stage uses to align the coronal fat-suppressed
    # series with the sagittal T2 the masks come from.
    pip("nibabel", "pydicom", "SimpleITK", "pandas")
    if INSTALL_SEGMENTATION:
        # Installed together so pip resolves ONE consistent set of versions. The old
        # notebook installed nnunetv2, then force-pinned an older nnunetv2 with
        # --no-deps on top, which left the environment internally inconsistent.
        print("python: segmentation stack (~5 min)")
        pip("nnunetv2>=2.8.1", "SPINEPS>=2.0.0", "totalspineseg>=20260623",
            "TotalSegmentator>=2.16.0")
    MARKER.touch()

print("\nversions:")
for mod in ("torch", "numpy", "nibabel", "nnunetv2", "spineps", "totalspineseg",
            "totalsegmentator"):
    try:
        m = importlib.import_module(mod)
        print(f"  {mod:18s} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"  {mod:18s} NOT IMPORTABLE — {str(exc)[:90]}")

try:
    import torch
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"\nGPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")
        if vram_gb >= 24:
            print("  Хватает на профиль QUALITY в ячейке 3 (кросс-проверка второй моделью")
            print("  + зеркальная TTA на устойчивость определения стороны).")
        else:
            print("  Профиль QUALITY лучше не включать: он рассчитан на 24+ ГБ VRAM.")
    else:
        print("\nNO GPU — Runtime → Change runtime type → GPU. Сегментация на CPU займёт часы.")
except Exception:
    pass

apt: dcm2niix, unzip
python packages: already installed in this VM (tick FORCE_REINSTALL to redo)

versions:
  torch              2.11.0+cu128
  numpy              2.0.2
  nibabel            5.4.2
  nnunetv2           NOT IMPORTABLE — No module named 'nnunetv2'
  spineps            NOT IMPORTABLE — No module named 'spineps'
  totalspineseg      NOT IMPORTABLE — No module named 'totalspineseg'
  totalsegmentator   NOT IMPORTABLE — No module named 'totalsegmentator'

GPU: NVIDIA A100-SXM4-80GB (85 GB)
  Хватает на профиль QUALITY в ячейке 3 (кросс-проверка второй моделью
  + зеркальная TTA на устойчивость определения стороны).


In [3]:
# @title 3 · Запуск пайплайна {display-mode:"form"}
DICOM_PATH = "https://raw.githubusercontent.com/omarnuri/MRI-reseqrch/main/OMER_NURIYEV%20(RAMIN)_dcm_8be27145-5512-4999-bb34-7e92b5f69749%20(1).zip"  # @param {type:"string"}
SUBJECT_ID = "anon"  # @param {type:"string"}
STAGES = "all"  # @param ["all", "ingest", "ingest,spineps", "register,fatsat_qc,facets_axial,posterior,marrow,report", "geometry,muscles,canal,discs,radiomics,report", "report"]
FORCE = ""  # @param {type:"string"}
QUALITY = True  # @param {type:"boolean"}
TTA_MIRROR = True  # @param {type:"boolean"}
SHOW_PHI_AUDIT = True  # @param {type:"boolean"}

# QUALITY: тратит GPU-время на надёжность, а не на скорость — независимая вторая
# модель позвонков (TotalSegmentator vertebrae_mr) даёт согласие по уровням.
# TTA_MIRROR: повторный прогон на зеркальной копии проверяет, устойчиво ли модель
# определяет лево/право. Ни то, ни другое не меняет определения измерений —
# только то, насколько мы знаем их надёжность. Для 24+ ГБ VRAM.

import json, sys
from pathlib import Path

for p in ("/content/spinelab-src",):
    if p not in sys.path:
        sys.path.insert(0, p)

from spinelab.config import DEFAULT_STAGES, Config
from spinelab.pipeline import run_pipeline

stages = DEFAULT_STAGES if STAGES == "all" else tuple(s.strip() for s in STAGES.split(",") if s.strip())
config = Config(
    dicom_source=DICOM_PATH,
    subject_id=SUBJECT_ID,
    work_dir=Path("/content/spine_work"),
    cache_dir=Path(CACHE_DIR),
    stages=stages,
    force=tuple(s.strip() for s in FORCE.split(",") if s.strip()),
    quality=QUALITY,
    tta_mirror=TTA_MIRROR,
)
results = run_pipeline(config)

print("\n" + "=" * 62)
for name, res in results.items():
    print(f"{name:18s} {res.status.value:9s} {res.reason or ''}"[:120])

if SHOW_PHI_AUDIT:
    audit = json.loads((config.results_dir / "phi_audit.json").read_text(encoding="utf-8")) \
        if (config.results_dir / "phi_audit.json").exists() else None
    if audit:
        print("\nИдентифицирующие теги в DICOM:")
        print(" ", audit["verdict"])
        print("  теги:", ", ".join(audit["phi_tags_present"]) or "—")
        print("  деидентифицировать:  !python -m spinelab deid --dicom <dir> --out /content/anon")

log: /content/spine_work/results/run.log
[ingest           ] cached — skipping (delete ingest.json to re-run)
[spineps          ] running…
[spineps          ] SKIP (0.0s) — spineps CLI not on PATH (pip install SPINEPS>=2.0.0)
[totalspineseg    ] running…
[totalspineseg    ] SKIP (0.0s) — totalspineseg CLI not on PATH (pip install totalspineseg)
[totalsegmentator ] running…
[totalsegmentator ] SKIP (0.0s) — TotalSegmentator not importable: No module named 'totalsegmentator'
[register         ] running…
[register         ] SKIP (0.0s) — no SPINEPS masks to move
[fatsat_qc        ] running…
[fatsat_qc        ] PARTIAL (1.2s) — NO evidence of fat suppression: band / core is 1.18 against 1.181 on the plain T2. Bright signal on this series should not be read as fluid or oedema
[crosscheck       ] running…
[crosscheck       ] SKIP (0.0s) — no SPINEPS instance mask to check against
[facets_axial     ] running…
[facets_axial     ] SKIP (0.0s) — need both SPINEPS instance and semantic masks
[geo

In [4]:
# @title 4 · Отчёт {display-mode:"form"}
COPY_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/spinelab-results"  # @param {type:"string"}

import shutil
from pathlib import Path
from IPython.display import HTML, display

report = Path("/content/spine_work/results/report.html")
if not report.exists():
    print("Отчёта нет — запустите ячейку 3 (этап report).")
else:
    if COPY_TO_DRIVE:
        dest = Path(DRIVE_RESULTS_DIR) / Path("/content/spine_work/results").name
        # Копируем только результаты (JSON/HTML/PNG), не промежуточные NIfTI:
        # они большие, а Drive-квота и канал ограничены.
        for src in Path("/content/spine_work/results").rglob("*"):
            if src.is_file() and src.suffix.lower() in (".html", ".json", ".png", ".csv"):
                out = dest / src.relative_to("/content/spine_work/results")
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, out)
        print(f"скопировано в {dest}")
    display(HTML(report.read_text(encoding="utf-8")))

скопировано в /content/drive/MyDrive/spinelab-results/results


### Быстрая проверка данных до запуска (10 секунд, без GPU)

```
!python -m spinelab inspect --dicom /content/drive/MyDrive/mri/study.zip
```

Печатает список серий (контраст, плоскость, срезы, TE/TR/TI) и вывод о том, на
какие вопросы это исследование может ответить, а на какие нет. Полезно посмотреть
до того, как тратить GPU.

### Порядок чтения отчёта

Сначала проверки, которые могут обесценить всё остальное: контуры сегментации →
подавление жира → устойчивость определения стороны → согласие двух моделей по
уровням. И только потом измерения. Подробно — `docs/COLAB.md`.

### Если что-то пошло не так

| Симптом | Что делать |
|---|---|
| Colab отключился на середине | Запустить ячейку 3 снова: выполненные этапы читаются из `results/stages/*.json`, продолжится с места обрыва. |
| Нужно пересчитать один этап | В поле `FORCE` через запятую: `marrow,posterior`. |
| `spineps` не импортируется | Ячейка 2 с `FORCE_REINSTALL`, затем Runtime → Restart session (вручную, один раз). |
| Веса качаются каждый раз | Проверьте, что `CACHE_DIR` в Drive и Drive смонтирован; ячейка 1 печатает размер кэша. |
| Этап `marrow` пропущен | В исследовании нет последовательности с подавлением жира — это ограничение данных, а не ошибка. См. `docs/clinical-context.md`. |
| Сравнение сторон «not_comparable» | Последовательность не покрывает обе стороны одинаково; разница была бы артефактом FOV. |
| `register` пропущен | Нет SimpleITK (ячейка 2) либо fat-sat серия и есть та же, что задаёт систему координат масок. Без него маски стоят по геометрии DICOM — работает, но без поправки на движение между сериями. |
| `register` не применил поправку | Оптимизатор дал неправдоподобный сдвиг (>15 мм или >10°) или не улучшил метрику — это признак неудачной регистрации, а не движения пациента. Оставляется выравнивание по заголовкам, и отчёт это пишет. |
| `crosscheck` пропущен | Профиль QUALITY выключен. |

Тесты (без GPU, ~1 с): `!python -m pytest /content/spinelab-src/tests -q`

In [5]:
import json, pathlib
res = pathlib.Path('/content/spine_work/results')
S = json.loads((res / 'summary.json').read_text(encoding='utf-8'))
F = json.loads((res / 'findings.json').read_text(encoding='utf-8'))

print('spinelab', S.get('spinelab_version'), '| failed:', S.get('failed'), '| skipped:', S.get('skipped'))
print('-' * 78)
for name, st in S['stages'].items():
    print(f"{name:17s} {st['status']:8s} {st.get('duration_s', 0):7.1f}s  {(st.get('reason') or '')[:80]}")

D = {k: (v.get('data') or {}) for k, v in F['stages'].items()}
def show(title, value):
    print(f"\n### {title}\n{json.dumps(value, ensure_ascii=False, indent=1)[:1200]}")

show('series', [[s.get('description'), s.get('sequence_label'), s.get('plane'), s.get('n_slices')]
                for s in D.get('ingest', {}).get('series', [])])
show('picks', D.get('ingest', {}).get('picks'))
show('fatsat_qc', {k: D.get('fatsat_qc', {}).get(k) for k in
                   ('suppression_effective', 'verdict', 'fatsat_stats', 'control_stats')})
show('register', {n: {'applied': t.get('applied'), **{k: t.get('registration', {}).get(k) for k in
                      ('translation_magnitude_mm', 'rotation_deg', 'reason')}}
                  for n, t in (D.get('register', {}).get('targets') or {}).items()})
show('mirror TTA', (D.get('spineps', {}).get('mirror_consistency') or {}).get('side_label_agreement'))
show('crosscheck', {k: D.get('crosscheck', {}).get(k) for k in
                    ('mean_dice', 'levels_needing_visual_check')})
show('geometry', {k: D.get('geometry', {}).get(k) for k in
                  ('levels_measured', 'max_wedge_angle_deg', 'longest_run_at_or_above_threshold',
                   'scheuermann_pattern', 'corpus_label_used', 'curvature', 'notes')})
show('wedges', [[v.get('name'), v.get('wedge_angle_deg'), v.get('anterior_height_mm'),
                 v.get('posterior_height_mm')] for v in D.get('geometry', {}).get('per_vertebra', [])])
show('facets_axial', {'largest': D.get('facets_axial', {}).get('largest_side_difference'),
                      'joints': [[j.get('joint'), j.get('comparable'),
                                  (j.get('sides', {}).get('left') or {}).get('bright_fraction'),
                                  (j.get('sides', {}).get('right') or {}).get('bright_fraction')]
                                 for j in D.get('facets_axial', {}).get('joints', [])]})
show('posterior', {g: {'status': e.get('status'), 'reason': e.get('reason'),
                       'bright': e.get('bright_fraction_comparison')}
                   for g, e in (D.get('posterior', {}).get('groups') or {}).items()})
show('marrow', {k: D.get('marrow', {}).get(k) for k in
                ('sequence_label', 'plane', 'n_measured', 'top_candidates',
                 'fat_suppression_verified')})
show('muscles', {m: v.get('comparison') for m, v in (D.get('muscles', {}).get('muscles') or {}).items()})
show('canal', {k: D.get('canal', {}).get(k) for k in
               ('median_area_mm2', 'p10_area_mm2', 'min_area_mm2', 'max_narrowing_pct')})
show('agreement', {k: D.get('agreement', {}).get(k) for k in ('dice', 'agreement_ok')})

spinelab 0.3.0 | failed: [] | skipped: ['spineps', 'totalspineseg', 'totalsegmentator', 'register', 'crosscheck', 'facets_axial', 'geometry', 'muscles', 'marrow', 'posterior', 'canal', 'discs', 'radiomics', 'agreement']
------------------------------------------------------------------------------
ingest            cached       0.0s  resumed from ingest.json (use force to re-run)
spineps           skipped      0.0s  spineps CLI not on PATH (pip install SPINEPS>=2.0.0)
totalspineseg     skipped      0.0s  totalspineseg CLI not on PATH (pip install totalspineseg)
totalsegmentator  skipped      0.0s  TotalSegmentator not importable: No module named 'totalsegmentator'
register          skipped      0.0s  no SPINEPS masks to move
fatsat_qc         partial      1.2s  NO evidence of fat suppression: band / core is 1.18 against 1.181 on the plain T
crosscheck        skipped      0.0s  no SPINEPS instance mask to check against
facets_axial      skipped      0.0s  need both SPINEPS instance and 